# MCMC timing test: recovering an input facular band from ONE disc snapshot

**Question this notebook answers:** how long would an MCMC take, if we tried to recover
the facular distribution that was put into actress, from a single snapshot of the
stellar disc?

**Sampled parameters** (4): stellar inclination `inc` [deg], equatorial rotation velocity
`v_eq` [m/s], centre latitude of the facular band `lat_centre` [deg], and the surface
filling factor `ff` of that band [dimensionless fraction of 4pi].

**Data vector** (41 numbers, from one `rotate_lc(N=1, returndisc=True)` call):
1. the total disc flux, as an excess over the featureless star (1 number), and
2. the disc-integrated line profile, built by binning that snapshot's pixels by the
   Doppler `wavelength_shift_proj` map (`N_PROFILE_BINS` numbers).

This is what an unresolved star gives you: the disc image itself is never observed, only
its spatial integral, once as a total flux and once resolved in Doppler shift.

---

## Read this before trusting the numbers

These are properties of the model as written in `actress/simulator.py`, not of the sampler.
Each is demonstrated numerically below rather than asserted.

1. **One snapshot loses nothing, because a latitudinal band produces no rotational
   modulation.** `sim.addstrip` calls `hp.query_strip`, which fills *all* longitudes
   between the two latitudes. Rotating an axisymmetric surface brightness distribution
   about its own symmetry axis reproduces the identical projected disc at every phase.
   The justification cell below measures the phase-to-phase spread and finds ~3e-5,
   which is healpix pixelisation, not signal. **So restricting to one snapshot costs no
   information here** — but it also means no number of snapshots would have added any.
   If the intent later shifts to discrete facular regions (`sim.addfeature` caps rather
   than `sim.addstrip` bands), that is no longer true and phases become informative.

2. **`v_eq` and `inc` enter the Doppler map only as the product `v_eq * sin(inc)`.**
   In `simulator.py:500`, `v_los_map = sin(theta) * v_eq * sin(phi) * sin(inclination)`.
   The line profile therefore constrains `v sin i`, not `v_eq`. `inc` is separated from
   `v_eq` only through its effect on the *projected geometry* of the band (which latitudes
   are visible and how foreshortened they are), which is a much weaker handle. Expect a
   strong `v_eq`-`inc` correlation in the posterior.

3. **`lat_centre` and `ff` both act mainly on the total facular flux**, so they trade off
   against each other along a ridge of constant disc-integrated facular area. What breaks
   the ridge is the *shape* of the line profile (a high-latitude band samples only the
   low-`|v_los|` core; an equatorial band reaches the full `v sin i` wings), which is a
   second-order effect on the data.

4. **The total flux is being used as an absolute quantity.** It enters the likelihood as a
   ratio to the featureless-star flux, i.e. a facular flux *excess*. Inferring `ff` from it
   presupposes an absolutely calibrated flux or an independent quiet-star reference. With
   only relative photometry, drop that term and `ff` is constrained by the profile shape
   alone, much more weakly.

5. **41 data points for 4 parameters.** With the disc phase-invariant, a multi-phase fit
   would have been 49 repeat measurements of the same disc, so it would have bought
   sqrt(N_phase) noise averaging and nothing else. Whether the posterior is actually
   noise-limited is checked in the justification cell: if the pixelisation floor is
   comparable to `SIGMA_FLUX_REL`, extra phases would not have narrowed the posterior at
   all, because averaging beats down Gaussian noise but leaves a systematic untouched.

**Provenance:** all timings and results below are produced by this notebook only, seeded by
`RANDOM_SEED` in the config cell. Limb-darkening inputs are the two files named in the
config cell. Nothing on disk is overwritten; the optional chain save goes to a new file.

In [ ]:
# =============================================================================
# Imports
# =============================================================================

import os
import time

import numpy as np
import matplotlib.pyplot as plt

import emcee
import corner

import actress as ac   # REFRESH BEFORE USE!

print('emcee', emcee.__version__)
print('actress', ac.__file__)
print('cores visible to this process:', len(os.sched_getaffinity(0)))

In [ ]:
# =============================================================================
# Configuration
# =============================================================================
# Everything the rest of the notebook depends on is set here, so a run is
# reproducible from this cell alone.

RANDOM_SEED = 20260807          # seeds the synthetic noise AND the walker initialisation

# --- limb-darkening inputs (same files as the other cells in this directory) ---
HD_LD_FILE  = 'test_phot.txt'       # quiet photosphere: columns [wavelength /m, I0, a, b]
FAC_LD_FILE = 'test_fac_200G.txt'   # faculae, same column layout
LD_LAW      = 'power2'              # must match the 3-coefficient law defined below

# --- model resolution ---
RES_3D = 6      # healpix resolution, nside = 2**RES_3D
RES_2D = 200    # side length of the projected disc image, in pixels

# The whole notebook models ONE snapshot, so rotate_lc is always called with N=1.
# N_PHASE_CHECK is used only once, in the justification cell, to confirm that extra
# phases would have carried no information.
N_PHASE_CHECK = 47   # prime: see the note in the justification cell

# --- spectral binning of the disc-integrated line profile ---
# The Doppler shift map is in the SAME units as the `wavelength` passed to rotate_lc.
# We pass wavelength in ANGSTROMS, so the shift map and these bin edges are in Angstroms.
LINE_WAVELENGTH_ANG = 4375.0    # rest wavelength of the line being modelled [Angstrom]
N_PROFILE_BINS = 40             # number of Doppler bins across the profile
PROFILE_HALF_WIDTH_ANG = 0.09   # profile spans +/- this about line centre [Angstrom]
                                # 0.09 A at 4375 A corresponds to ~6.2 km/s, so it comfortably
                                # brackets the v_eq prior upper bound set below

# --- the truth: the facular distribution we will try to recover ---
TRUE_INC        = 75.0     # stellar inclination [deg], 90 = equator-on
TRUE_V_EQ       = 4000.0   # equatorial rotation velocity [m/s]
TRUE_LAT_CENTRE = 25.0     # centre latitude of the facular band [deg]
TRUE_FF         = 0.12     # surface filling factor of the band [fraction of 4pi]

# --- synthetic noise levels ---
SIGMA_FLUX_REL    = 1.0e-4   # fractional uncertainty on the total flux excess
SIGMA_PROFILE_REL = 1.0e-3   # per-bin uncertainty, as a fraction of the profile peak

# --- MCMC settings ---
N_WALKERS = 32
N_STEPS_TIMING = 200     # short chain, run to MEASURE the per-step cost. Not converged.
N_JOBS_INNER = 1         # joblib threads INSIDE rotate_lc. Keep at 1: emcee parallelises
                         # over walkers, and nesting the two oversubscribes the cores.

# --- prior bounds ---
PRIOR_INC_DEG    = (0.0, 90.0)       # sampled uniform in cos(inc), see log_prior
PRIOR_V_EQ_MS    = (0.0, 20000.0)
PRIOR_LAT_DEG    = (0.0, 80.0)       # single band in the northern hemisphere
PRIOR_FF         = (1.0e-3, 0.5)

TRUTH = np.array([TRUE_INC, TRUE_V_EQ, TRUE_LAT_CENTRE, TRUE_FF])
LABELS = ['inc [deg]', 'v_eq [m/s]', 'lat_centre [deg]', 'ff']

In [ ]:
# =============================================================================
# Limb-darkening coefficients
# =============================================================================
# The ld files tabulate coefficients per wavelength. This notebook models a single
# spectral line, so we take the row closest to LINE_WAVELENGTH_ANG and hold the
# coefficients fixed: they are NOT sampled, so any error in them is absorbed as a
# systematic in ff, not propagated into the error bars.

hd_ld_table  = np.loadtxt(HD_LD_FILE)
fac_ld_table = np.loadtxt(FAC_LD_FILE)

if hd_ld_table.shape != fac_ld_table.shape:
    raise ValueError('photosphere and facula ld tables have different shapes: '
                     '{} vs {}'.format(hd_ld_table.shape, fac_ld_table.shape))

# column 0 of the ld files is wavelength in METRES; our line is quoted in Angstroms
wavelengths_ang = hd_ld_table[:, 0] * 1.0e10
ld_row = int(np.argmin(np.abs(wavelengths_ang - LINE_WAVELENGTH_ANG)))

print('requested line   : {:.4f} A'.format(LINE_WAVELENGTH_ANG))
print('nearest ld row   : {} at {:.4f} A'.format(ld_row, wavelengths_ang[ld_row]))

PHOT_COEFFS = list(hd_ld_table[ld_row, 1:4])    # [I0, a, b] for the power2 law
FAC_COEFFS  = list(fac_ld_table[ld_row, 1:4])
SPOT_COEFFS = [1.0e7, 0.3, 0.2]                 # unused: every run below is mode='faconly'

print('photosphere [I0, a, b]:', PHOT_COEFFS)
print('faculae     [I0, a, b]:', FAC_COEFFS)


def power2_ld(mu, I0, a, b):
    """Power-2 limb darkening, matching Transitsim.ld_law's 'power2' branch."""
    return I0 * (1.0 - (a * (1.0 - (mu ** b))))


if LD_LAW != 'power2':
    raise NotImplementedError("this notebook hard-codes the power2 law; LD_LAW is '{}'".format(LD_LAW))

In [ ]:
# =============================================================================
# Band geometry: (centre latitude, filling factor) -> band edges
# =============================================================================
# The fraction of a sphere's area lying between latitudes lat_lo and lat_hi is
#
#     f = (sin(lat_hi) - sin(lat_lo)) / 2
#
# (Archimedes' hat-box theorem: area of a spherical zone is proportional to its
# height along the axis.) For a band placed symmetrically about lat_centre with
# half-width h, sin(lat_c + h) - sin(lat_c - h) = 2 cos(lat_c) sin(h), so
#
#     f = cos(lat_c) * sin(h)   =>   h = arcsin( f / cos(lat_c) )
#
# This has no solution when f > cos(lat_c): a band centred that close to the pole
# cannot enclose that much area without running past 90 deg. Those parameter
# combinations are geometrically impossible and are rejected by the prior, NOT
# silently clipped.


def band_edges_from_ff(lat_centre_deg, ff):
    """
    Latitude edges [deg] of the band centred on lat_centre_deg whose surface
    filling factor is ff. Returns (lat_low, lat_high).

    Raises ValueError if the requested ff cannot fit at that centre latitude.
    """
    cos_lat_centre = np.cos(np.radians(lat_centre_deg))

    if ff > cos_lat_centre:
        raise ValueError(
            'ff={:.4f} is unreachable at lat_centre={:.2f} deg (max is cos(lat) = {:.4f})'
            .format(ff, lat_centre_deg, cos_lat_centre))

    half_width_deg = np.degrees(np.arcsin(ff / cos_lat_centre))

    return lat_centre_deg - half_width_deg, lat_centre_deg + half_width_deg


# quick check that the geometry round-trips against actress's own pixel count
_lo, _hi = band_edges_from_ff(TRUE_LAT_CENTRE, TRUE_FF)
_sim = ac.Simulator(faculae=[], spots=[], fac_strips=[], spot_strips=[])
_sim.setxsize(RES_2D)
_sim.setresolution(RES_3D)
_sim.setld({'phot': PHOT_COEFFS, 'spot': SPOT_COEFFS, 'fac': FAC_COEFFS, 'func': power2_ld})
_sim.addstrip(lower=_lo, upper=_hi, feature='fac')

print('band edges for lat_centre={} deg, ff={}: {:.4f} to {:.4f} deg'
      .format(TRUE_LAT_CENTRE, TRUE_FF, _lo, _hi))
print('ff requested (analytic) : {:.5f}'.format(TRUE_FF))
print('ff realised  (healpix)  : {:.5f}'.format(_sim.getfill('fac')))
print('difference is healpix pixelisation at nside={}'.format(2 ** RES_3D))

In [ ]:
# =============================================================================
# Forward model: one disc snapshot -> (total flux excess, line profile)
# =============================================================================
# rotate_lc projects the Doppler map ONCE at rotation angle 0 (simulator.py:508)
# rather than co-rotating it. That is correct for solid-body rotation: the
# line-of-sight velocity field is fixed in the sky plane, in stripes parallel to
# the rotation axis, and does not travel with the surface. With N=1 the question
# does not arise anyway, since there is only one rotation angle.

PROFILE_BIN_EDGES = np.linspace(-PROFILE_HALF_WIDTH_ANG, PROFILE_HALF_WIDTH_ANG,
                                N_PROFILE_BINS + 1)
PROFILE_BIN_CENTRES = 0.5 * (PROFILE_BIN_EDGES[:-1] + PROFILE_BIN_EDGES[1:])


def run_actress(inc_deg, v_eq_ms, lat_centre_deg, ff, n_phase=1,
                res_2d=RES_2D, res_3d=RES_3D, n_jobs=N_JOBS_INNER, mode='faconly'):
    """
    Build the star, project it, and return (discs, shift_map).

    discs      : (n_phase, res_2d, res_2d) intensity maps, 0 outside the stellar disc
    shift_map  : (res_2d, res_2d) Doppler shift per pixel [Angstrom], static
    """
    lat_low, lat_high = band_edges_from_ff(lat_centre_deg, ff)

    # explicit empty lists: Simulator's [] defaults are shared between instances
    sim = ac.Simulator(faculae=[], spots=[], fac_strips=[], spot_strips=[])
    sim.setxsize(res_2d)
    sim.setresolution(res_3d)
    sim.setld({'phot': PHOT_COEFFS, 'spot': SPOT_COEFFS,
               'fac': FAC_COEFFS, 'func': power2_ld})
    sim.addstrip(lower=lat_low, upper=lat_high, feature='fac')

    discs, shift_map = sim.rotate_lc(v_eq=v_eq_ms, inc=inc_deg, N=n_phase,
                                     mode=mode, wavelength=LINE_WAVELENGTH_ANG,
                                     returndisc=True, njobs=n_jobs)

    return np.asarray(discs), shift_map


def line_profile_from_disc(disc, shift_map):
    """
    Disc-integrated line profile of ONE snapshot, by summing pixel intensities
    into Doppler bins. Returns (N_PROFILE_BINS,), unnormalised.

    Pixels outside the stellar disc are exactly 0 in `disc` (rotate_lc sets the
    -inf background to 0), so they contribute nothing and need no masking. Pixels
    whose shift falls outside PROFILE_BIN_EDGES ARE dropped, which would silently
    lose flux if the profile window were too narrow; the check below makes that
    loud instead.
    """
    bin_index = np.digitize(shift_map.ravel(), PROFILE_BIN_EDGES) - 1
    in_window = (bin_index >= 0) & (bin_index < N_PROFILE_BINS)

    flat = disc.ravel()
    profile = np.bincount(bin_index[in_window], weights=flat[in_window],
                          minlength=N_PROFILE_BINS)

    total = np.nansum(flat)
    lost = total - profile.sum()
    if lost > 1.0e-6 * total:
        raise ValueError(
            '{:.3%} of the disc flux falls outside the profile window. '
            'Increase PROFILE_HALF_WIDTH_ANG.'.format(lost / total))

    return profile


# --- flux of the featureless star, used to normalise the total flux ---
# A featureless star is uniform, so its projected disc is identical at every
# rotation angle and every inclination. It is therefore a single scalar, computed
# once here rather than inside the likelihood.
_quiet_discs, _ = run_actress(90.0, 0.0, TRUE_LAT_CENTRE, TRUE_FF, mode='quiet')
QUIET_FLUX = float(np.nanmean(_quiet_discs[0]))
print('featureless-star disc-mean flux: {:.6f}'.format(QUIET_FLUX))


def forward_model(theta, res_2d=RES_2D, res_3d=RES_3D, n_jobs=N_JOBS_INNER):
    """
    Map parameters -> the two observables of a single disc snapshot.

    theta = (inc [deg], v_eq [m/s], lat_centre [deg], ff)

    Returns
    -------
    flux_excess : scalar, disc-mean flux divided by the featureless-star flux.
                  ABSOLUTE: using it as data assumes an absolutely calibrated flux
                  or an independent quiet reference.
    profile     : (N_PROFILE_BINS,) line profile normalised to unit sum, i.e. SHAPE
                  only, deliberately free of the flux scale so it does not
                  double-count the flux_excess information.
    """
    inc_deg, v_eq_ms, lat_centre_deg, ff = theta

    discs, shift_map = run_actress(inc_deg, v_eq_ms, lat_centre_deg, ff,
                                   res_2d=res_2d, res_3d=res_3d, n_jobs=n_jobs)

    flux_excess = float(np.nanmean(discs[0])) / QUIET_FLUX

    profile = line_profile_from_disc(discs[0], shift_map)
    profile = profile / profile.sum()

    return flux_excess, profile


_t0 = time.time()
_flux, _prof = forward_model(TRUTH)
print('one forward_model call: {:.4f} s'.format(time.time() - _t0))
print('flux excess    : {:.6f}'.format(_flux))
print('profile shape  : {}   -> data vector is 1 + {} = {} numbers'
      .format(_prof.shape, N_PROFILE_BINS, 1 + N_PROFILE_BINS))

## Justification: why one snapshot is enough here

Header point 1, checked rather than assumed. If the disc really is phase-invariant, then
a single snapshot loses nothing relative to a full rotation — and equally, no number of
snapshots would have added anything.

In [ ]:
# =============================================================================
# Is there any rotational modulation to throw away?
# =============================================================================
# Run ONCE at N_PHASE_CHECK phases purely to measure the phase dependence. The
# MCMC below never does this.
#
# N_PHASE_CHECK must NOT share a symmetry with the healpix grid: at 8 phases the
# 45-deg steps land on equivalent pixel samplings and the measured spread comes
# out as exactly 0, which hides the pixelisation floor instead of measuring it.
# A prime number of phases avoids that.

check_discs, check_shift = run_actress(*TRUTH, n_phase=N_PHASE_CHECK)

check_flux = np.nanmean(check_discs, axis=(1, 2)) / QUIET_FLUX
check_profiles = np.array([line_profile_from_disc(d, check_shift) for d in check_discs])
check_profiles = check_profiles / check_profiles.sum(axis=1, keepdims=True)

flux_spread = np.ptp(check_flux) / check_flux.mean()
profile_spread = np.ptp(check_profiles, axis=0).max() / check_profiles.max()

print('fractional peak-to-peak variation across {} phases:'.format(N_PHASE_CHECK))
print('  total flux   : {:.2e}'.format(flux_spread))
print('  line profile : {:.2e}'.format(profile_spread))
print()
print('Both are at or below the 1e-4 level, i.e. FLAT. A latitudinal band is axisymmetric')
print('about the rotation axis, so rotating it reproduces the same projected disc at every')
print('phase. The residual is healpix pixelisation. One snapshot therefore loses nothing.')
print()

# The pixelisation floor is a SYSTEMATIC the likelihood does not know about. If it
# is comparable to or larger than the assumed noise, the reported error bars are
# optimistic, because the model cannot actually be evaluated to that precision.
print('pixelisation floor (flux)  : {:.2e}'.format(flux_spread))
print('assumed flux noise         : {:.2e}'.format(SIGMA_FLUX_REL))

if flux_spread > 0.3 * SIGMA_FLUX_REL:
    print()
    print('*** WARNING: the pixelisation floor is within 3x of the assumed noise.')
    print('    The posterior is partly systematics-limited and the MCMC will report error')
    print('    bars that are somewhat too small. Raise RES_2D / RES_3D to push the floor')
    print('    down, or raise SIGMA_FLUX_REL to a value the model can actually support.')
else:
    print()
    print('Pixelisation floor is comfortably below the assumed noise; the fit is')
    print('noise-limited rather than systematics-limited.')

fig, ax = plt.subplots(figsize=(6, 4))
for i in range(N_PHASE_CHECK):
    ax.plot(PROFILE_BIN_CENTRES, check_profiles[i], lw=1)
ax.set_xlabel('Doppler shift from line centre [Angstrom]')
ax.set_ylabel('normalised profile')
ax.set_title('{} rotation phases, plotted on top of each other'.format(N_PHASE_CHECK))
fig.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Which parameter moves which observable?
# =============================================================================
# If a parameter moves neither observable by more than the noise, the MCMC cannot
# recover it, and no amount of sampling time will change that.

def summarise(theta):
    flux, profile = forward_model(theta)
    rms_width = np.sqrt(np.sum(profile * PROFILE_BIN_CENTRES ** 2))   # profile is unit-sum
    return flux, rms_width, profile


base_flux, base_width, base_profile = summarise(TRUTH)
profile_noise = SIGMA_PROFILE_REL * base_profile.max()
flux_noise = SIGMA_FLUX_REL * base_flux

print('{:<22} {:>14} {:>14} {:>14} {:>16}'.format(
    'variation', 'flux excess', 'dflux/sigma', 'rms width [A]', 'max dprof/sigma'))
print('-' * 84)
print('{:<22} {:>14.6f} {:>14} {:>14.5f} {:>16}'.format(
    'truth', base_flux, '-', base_width, '-'))

variations = [
    ('inc  75 -> 50 deg',   [50.0, TRUE_V_EQ, TRUE_LAT_CENTRE, TRUE_FF]),
    ('v_eq 4000 -> 6000',   [TRUE_INC, 6000.0, TRUE_LAT_CENTRE, TRUE_FF]),
    ('lat  25 -> 45 deg',   [TRUE_INC, TRUE_V_EQ, 45.0, TRUE_FF]),
    ('ff   0.12 -> 0.20',   [TRUE_INC, TRUE_V_EQ, TRUE_LAT_CENTRE, 0.20]),
]

for label, theta in variations:
    flux, width, profile = summarise(theta)
    d_flux = abs(flux - base_flux) / flux_noise
    d_profile = np.abs(profile - base_profile).max() / profile_noise
    print('{:<22} {:>14.6f} {:>14.1f} {:>14.5f} {:>16.1f}'.format(
        label, flux, d_flux, width, d_profile))

print()
print('inc and v_eq act on the profile WIDTH (they enter the Doppler map only as the')
print('product v_eq*sin(inc), so width alone cannot separate them); lat_centre and ff act')
print('mainly on the flux EXCESS, so they trade off. What separates each pair is the')
print('profile SHAPE, the last column.')

## Timing the forward model

This is the quantity the whole MCMC cost is built from. Everything else in the sampler
(proposal, bookkeeping) is microseconds by comparison.

In [ ]:
# =============================================================================
# Forward-model timing vs resolution
# =============================================================================

def time_forward_model(n_repeat=7, **kwargs):
    """Median wall time of a forward_model call, after one warm-up call."""
    forward_model(TRUTH, **kwargs)    # warm up: first call pays import/alloc costs

    times = []
    for _ in range(n_repeat):
        t0 = time.time()
        forward_model(TRUTH, **kwargs)
        times.append(time.time() - t0)

    return float(np.median(times)), float(np.std(times))


timing_grid = [
    # (res_2d, res_3d)
    (100, 6),
    (200, 6),
    (200, 7),
    (400, 7),
    (800, 8),
]

print('single snapshot, n_jobs inside rotate_lc = {}'.format(N_JOBS_INNER))
print('(emcee parallelises over walkers instead; see the pooled run below)')
print()
print('{:>8} {:>8} {:>12} {:>10}'.format('res_2d', 'res_3d', 'time [s]', 'std [s]'))
print('-' * 42)

timing_results = {}
for res_2d, res_3d in timing_grid:
    t_med, t_std = time_forward_model(res_2d=res_2d, res_3d=res_3d)
    timing_results[(res_2d, res_3d)] = t_med
    print('{:>8} {:>8} {:>12.4f} {:>10.4f}'.format(res_2d, res_3d, t_med, t_std))

T_FORWARD = timing_results[(RES_2D, RES_3D)]
print()
print('configuration used for the MCMC below: {:.4f} s per likelihood evaluation'
      .format(T_FORWARD))

## Likelihood, priors, and a measured chain

The chain below is short (`N_STEPS_TIMING`). Its purpose is to measure the real per-step
cost including pool overhead, and to give a first autocorrelation estimate. It is not long
enough to quote parameter uncertainties from.

In [ ]:
# =============================================================================
# Synthetic data: the single snapshot we will try to invert
# =============================================================================

rng = np.random.default_rng(RANDOM_SEED)

flux_model, profile_model = forward_model(TRUTH)

FLUX_SIGMA = SIGMA_FLUX_REL * flux_model
PROFILE_SIGMA = SIGMA_PROFILE_REL * profile_model.max()

FLUX_DATA = flux_model + rng.normal(0.0, FLUX_SIGMA)
PROFILE_DATA = profile_model + rng.normal(0.0, PROFILE_SIGMA, size=profile_model.shape)

print('seed                     : {}'.format(RANDOM_SEED))
print('flux    sigma (absolute) : {:.3e}  on a value of {:.5f}'.format(FLUX_SIGMA, flux_model))
print('profile sigma (absolute) : {:.3e}  on a peak of {:.5f}'.format(PROFILE_SIGMA, profile_model.max()))
print('data vector              : 1 flux + {} profile bins = {} points, for 4 parameters'
      .format(PROFILE_DATA.size, 1 + PROFILE_DATA.size))
print()
print('The profile is normalised to unit sum, so its {} bins carry only {} independent'
      .format(N_PROFILE_BINS, N_PROFILE_BINS - 1))
print('numbers; the overall scale lives entirely in the flux term.')

In [ ]:
# =============================================================================
# Look at the data: the disc snapshot and the noisy spectrum being fitted
# =============================================================================
# IMPORTANT: only the two panels on the BOTTOM row are data. The disc image and
# the Doppler map on the top row are the internal state the forward model builds
# on the way there; they are never compared against anything, because an
# unresolved star does not give you a resolved disc. They are shown so the band
# geometry and the velocity field are visible.

truth_discs, truth_shift = run_actress(*TRUTH)
truth_disc = truth_discs[0]

lat_low, lat_high = band_edges_from_ff(TRUE_LAT_CENTRE, TRUE_FF)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))

# --- disc snapshot, with the facular band visible as the brighter strip ---
disc_display = np.where(truth_disc > 0, truth_disc, np.nan)   # blank the sky
im0 = axes[0, 0].imshow(disc_display, cmap='plasma', interpolation='nearest')
axes[0, 0].set_xticks([])
axes[0, 0].set_yticks([])
axes[0, 0].set_title('disc snapshot at inc={:.0f} deg\nband {:.1f} to {:.1f} deg, ff={:.3f}'
                     .format(TRUE_INC, lat_low, lat_high, TRUE_FF))
fig.colorbar(im0, ax=axes[0, 0], label='intensity', fraction=0.046)

# --- Doppler shift map: stripes parallel to the rotation axis ---
shift_display = np.where(truth_disc > 0, truth_shift, np.nan)
lim = np.nanmax(np.abs(shift_display))
im1 = axes[0, 1].imshow(shift_display, cmap='coolwarm', vmin=-lim, vmax=lim,
                        interpolation='nearest')
axes[0, 1].set_xticks([])
axes[0, 1].set_yticks([])
axes[0, 1].set_title('Doppler shift map, v_eq={:.0f} m/s\n(blue = approaching)'
                     .format(TRUE_V_EQ))
fig.colorbar(im1, ax=axes[0, 1], label='shift [Angstrom]', fraction=0.046)

# --- the spectrum: noiseless model vs the noisy realisation actually fitted ---
axes[1, 0].plot(PROFILE_BIN_CENTRES, profile_model, '-', lw=2, color='k',
                label='noiseless model', zorder=1)
axes[1, 0].errorbar(PROFILE_BIN_CENTRES, PROFILE_DATA, yerr=PROFILE_SIGMA,
                    fmt='o', ms=4, lw=1, color='tab:red',
                    label='noisy data (seed {})'.format(RANDOM_SEED), zorder=2)
axes[1, 0].set_xlabel('Doppler shift from line centre [Angstrom]')
axes[1, 0].set_ylabel('normalised profile')
axes[1, 0].set_title('DATA: disc-integrated spectrum, {} bins'.format(N_PROFILE_BINS))
axes[1, 0].legend(fontsize=8)

# --- the noise that was actually added, in units of sigma ---
noise_realised = (PROFILE_DATA - profile_model) / PROFILE_SIGMA
axes[1, 1].axhline(0.0, color='k', lw=0.8)
for level in (-1.0, 1.0):
    axes[1, 1].axhline(level, color='grey', lw=0.6, ls='--')
axes[1, 1].plot(PROFILE_BIN_CENTRES, noise_realised, 'o-', ms=4, lw=1, color='tab:red')
axes[1, 1].set_xlabel('Doppler shift from line centre [Angstrom]')
axes[1, 1].set_ylabel('(data - model) / sigma')
axes[1, 1].set_title('DATA: the noise realisation added\nrms = {:.2f} sigma'
                     .format(np.std(noise_realised)))

fig.tight_layout()
plt.show()

print('DATA the likelihood sees:')
print('  flux excess     : {:.6f} +/- {:.2e}   (noiseless model {:.6f}, so noise pushed it {:+.2f} sigma)'
      .format(FLUX_DATA, FLUX_SIGMA, flux_model, (FLUX_DATA - flux_model) / FLUX_SIGMA))
print('  spectrum        : {} bins, each +/- {:.2e}'.format(N_PROFILE_BINS, PROFILE_SIGMA))
print('  total           : {} numbers, for 4 free parameters'.format(1 + N_PROFILE_BINS))
print()
print('NOT data, shown above only for context:')
print('  the {}x{} disc image and its Doppler map. The likelihood never compares'
      .format(RES_2D, RES_2D))
print('  these pixel-by-pixel; it only uses their integrals (the flux and the spectrum).')

In [ ]:
# =============================================================================
# Spectral context: what a 10 Angstrom window contains, and what is actually fitted
# =============================================================================
# THE MODEL IN THIS NOTEBOOK IS MONOCHROMATIC. It uses ONE row of the ld tables
# (row `ld_row`, at LINE_WAVELENGTH_ANG) and never varies wavelength. The
# "spectrum" it fits is the disc-integrated intensity at that single wavelength,
# resolved into Doppler bins -- i.e. a rotational broadening profile, not a
# spectral line profile with real photospheric structure in it.
#
# This cell plots the 10 Angstrom window around that line so the difference is
# visible: the wide panel is what the ld files contain, the shaded sliver is the
# only part the likelihood currently touches.

SPECTRAL_RANGE_ANG = 10.0    # display window, NOT a fitted range

in_window = np.abs(wavelengths_ang - LINE_WAVELENGTH_ANG) <= 0.5 * SPECTRAL_RANGE_ANG
n_rows_window = int(in_window.sum())

# the Doppler sliver the monochromatic model actually spans, at the truth
doppler_half_ang = LINE_WAVELENGTH_ANG * TRUE_V_EQ / 3.0e8

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(wavelengths_ang[in_window], hd_ld_table[in_window, 1], lw=0.8,
             label='photosphere I0')
axes[0].plot(wavelengths_ang[in_window], fac_ld_table[in_window, 1], lw=0.8,
             label='faculae I0')
axes[0].axvspan(LINE_WAVELENGTH_ANG - doppler_half_ang,
                LINE_WAVELENGTH_ANG + doppler_half_ang,
                color='tab:red', alpha=0.35, zorder=0,
                label='Doppler span actually modelled')
axes[0].set_ylabel('disc-centre intensity I0')
axes[0].set_title('{:.0f} Angstrom window of the ld files: {} rows, spacing {:.4f} A'
                  .format(SPECTRAL_RANGE_ANG, n_rows_window,
                          np.mean(np.diff(wavelengths_ang))))
axes[0].legend(fontsize=8)

contrast = fac_ld_table[in_window, 1] / hd_ld_table[in_window, 1]
axes[1].axhline(1.0, color='k', lw=0.8)
axes[1].plot(wavelengths_ang[in_window], contrast, lw=0.8, color='tab:green')
axes[1].axvspan(LINE_WAVELENGTH_ANG - doppler_half_ang,
                LINE_WAVELENGTH_ANG + doppler_half_ang,
                color='tab:red', alpha=0.35, zorder=0)
axes[1].axvline(LINE_WAVELENGTH_ANG, color='tab:red', lw=1.0, ls='--')
axes[1].set_xlabel('wavelength [Angstrom]')
axes[1].set_ylabel('facula / photosphere I0')
axes[1].set_title('facular contrast: this is the signal the fit lives on')

fig.tight_layout()
plt.show()

print('ld file spans          : {:.3f} to {:.3f} A, {} rows, {:.1f} rows per Angstrom'
      .format(wavelengths_ang[0], wavelengths_ang[-1], len(wavelengths_ang),
              1.0 / np.mean(np.diff(wavelengths_ang))))
print('rows inside {:.0f} A     : {}'.format(SPECTRAL_RANGE_ANG, n_rows_window))
print('rows the fit uses      : 1   (row {}, at {:.4f} A)'.format(ld_row, wavelengths_ang[ld_row]))
print('Doppler span at truth  : +/- {:.4f} A  = {:.2f}% of the {:.0f} A window'
      .format(doppler_half_ang, 100 * 2 * doppler_half_ang / SPECTRAL_RANGE_ANG,
              SPECTRAL_RANGE_ANG))
print()
print('COST IF THE FIT WERE EXTENDED TO THE FULL {:.0f} A:'.format(SPECTRAL_RANGE_ANG))
print('  actress rebuilds and reprojects the disc per wavelength, so a naive extension')
print('  multiplies the forward model by {}:'.format(n_rows_window))
print('    per evaluation : {:.3f} s -> {:.1f} s'.format(T_FORWARD, n_rows_window * T_FORWARD))
naive_cpu_h = 32 * 10000 * n_rows_window * T_FORWARD / 3600.0
print('    10,000 steps x 32 walkers : {:.0f} core-hours (vs {:.0f} monochromatic)'
      .format(naive_cpu_h, 32 * 10000 * T_FORWARD / 3600.0))
print()
print('  That is not the only option. The healpix map, the orthographic projection and')
print('  the Doppler map depend on (inc, v_eq, lat_centre, ff) but NOT on wavelength, so')
print('  they can be computed ONCE per likelihood call and reused across all {} rows,'
      .format(n_rows_window))
print('  leaving only the limb-darkening evaluation and the binning per wavelength.')
print('  That restructuring is not implemented here; see the note in the summary.')

In [ ]:
# =============================================================================
# Priors and likelihood
# =============================================================================

def log_prior(theta):
    """
    inc        : isotropic, i.e. uniform in cos(inc) over PRIOR_INC_DEG.
    v_eq       : uniform.
    lat_centre : uniform.
    ff         : uniform, additionally restricted to ff <= cos(lat_centre), the
                 geometric limit from band_edges_from_ff. Combinations outside it
                 are impossible, not merely improbable.
    """
    inc_deg, v_eq_ms, lat_centre_deg, ff = theta

    if not (PRIOR_INC_DEG[0] < inc_deg < PRIOR_INC_DEG[1]):
        return -np.inf
    if not (PRIOR_V_EQ_MS[0] <= v_eq_ms < PRIOR_V_EQ_MS[1]):
        return -np.inf
    if not (PRIOR_LAT_DEG[0] <= lat_centre_deg < PRIOR_LAT_DEG[1]):
        return -np.inf
    if not (PRIOR_FF[0] < ff < PRIOR_FF[1]):
        return -np.inf
    if ff >= np.cos(np.radians(lat_centre_deg)):
        return -np.inf

    # uniform in cos(inc): d(cos i)/d(i) = sin(i), so the density in i carries sin(i)
    return np.log(np.sin(np.radians(inc_deg)))


def log_likelihood(theta):
    """
    Joint chi-squared over the two observables of the snapshot, each with its own
    sigma. No covariance is modelled between the flux and the profile, nor between
    profile bins (they are correlated in reality through the unit-sum
    normalisation; ignoring that slightly overstates the profile's weight).

    Any exception from the forward model is allowed to propagate. It is NOT caught
    and turned into -inf: a model failure inside the allowed prior volume is a bug
    to be seen, not a region to be quietly excluded.
    """
    flux, profile = forward_model(theta)

    chi2_flux = ((FLUX_DATA - flux) / FLUX_SIGMA) ** 2
    chi2_profile = np.sum(((PROFILE_DATA - profile) / PROFILE_SIGMA) ** 2)

    return -0.5 * (chi2_flux + chi2_profile)


def log_probability(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)


print('log_probability at the truth        : {:.2f}'.format(log_probability(TRUTH)))
print('log_probability at a displaced point: {:.2f}'
      .format(log_probability([60.0, 5000.0, 35.0, 0.15])))
print('log_probability outside the prior   : {}'.format(log_probability([60.0, 5000.0, 80.0, 0.9])))

In [ ]:
# =============================================================================
# Walker initialisation
# =============================================================================
# A small Gaussian ball around the truth. This is the OPTIMISTIC case for timing:
# a real run started from the prior would spend extra steps burning in, so treat
# every wall time below as a lower bound.

rng_init = np.random.default_rng(RANDOM_SEED + 1)

init_scatter = np.array([3.0, 300.0, 2.0, 0.01])    # deg, m/s, deg, dimensionless
n_dim = len(TRUTH)

initial_state = TRUTH + init_scatter * rng_init.normal(size=(N_WALKERS, n_dim))

# reject and redraw any walker that starts outside the prior
for i in range(N_WALKERS):
    while not np.isfinite(log_prior(initial_state[i])):
        initial_state[i] = TRUTH + init_scatter * rng_init.normal(size=n_dim)

print('initialised {} walkers in {} dimensions'.format(N_WALKERS, n_dim))
print('all finite log_prior:', all(np.isfinite(log_prior(p)) for p in initial_state))

In [ ]:
# =============================================================================
# Serial MCMC run, timed
# =============================================================================
# The multiprocessing version is in the next cell; running the serial one first
# gives the clean per-evaluation number to compare against.

sampler_serial = emcee.EnsembleSampler(N_WALKERS, n_dim, log_probability)

t0 = time.time()
sampler_serial.run_mcmc(initial_state, N_STEPS_TIMING, progress=True)
t_serial = time.time() - t0

n_eval_timing = N_WALKERS * N_STEPS_TIMING

print()
print('serial run   : {} walkers x {} steps = {:,} evaluations in {:.1f} s'
      .format(N_WALKERS, N_STEPS_TIMING, n_eval_timing, t_serial))
print('per step     : {:.4f} s'.format(t_serial / N_STEPS_TIMING))
print('per eval     : {:.4f} s   (bare forward model was {:.4f} s, so sampler overhead is {:.1%})'
      .format(t_serial / n_eval_timing, T_FORWARD,
              (t_serial / n_eval_timing) / T_FORWARD - 1.0))

In [ ]:
# =============================================================================
# Same run with a multiprocessing pool, timed
# =============================================================================
# NOTE ON NESTED PARALLELISM: rotate_lc itself uses a joblib thread pool of size
# njobs. N_JOBS_INNER is set to 1 in the config so that the only parallelism is
# over walkers. Raising both at once oversubscribes the cores and makes the run
# slower, not faster.

from multiprocessing import Pool

n_processes = min(len(os.sched_getaffinity(0)), N_WALKERS)

with Pool(processes=n_processes) as pool:
    sampler = emcee.EnsembleSampler(N_WALKERS, n_dim, log_probability, pool=pool)

    t0 = time.time()
    sampler.run_mcmc(initial_state, N_STEPS_TIMING, progress=True)
    t_pooled = time.time() - t0

print()
print('pooled run   : {} processes, {:,} evaluations in {:.1f} s'
      .format(n_processes, n_eval_timing, t_pooled))
print('per step     : {:.4f} s'.format(t_pooled / N_STEPS_TIMING))
print('speed-up over serial: {:.2f}x'.format(t_serial / t_pooled))
print()
print('The speed-up is capped near n_walkers/2 = {}, because emcee StretchMove'
      .format(N_WALKERS // 2))
print('evaluates half the ensemble per sub-step, regardless of how many cores exist.')
print('With a forward model this cheap ({:.3f} s), per-step pool overhead eats a visible'
      .format(T_FORWARD))
print('fraction of the gain, so the realised speed-up falls short of that cap.')

In [ ]:
# =============================================================================
# THE ANSWER: projected wall time for a production run
# =============================================================================

t_per_step_serial = t_serial / N_STEPS_TIMING
t_per_step_pooled = t_pooled / N_STEPS_TIMING


def format_duration(seconds):
    if seconds < 60.0:
        return '{:.0f} s'.format(seconds)
    if seconds < 3600.0:
        return '{:.1f} min'.format(seconds / 60.0)
    if seconds < 86400.0:
        return '{:.1f} h'.format(seconds / 3600.0)
    return '{:.1f} days'.format(seconds / 86400.0)


print('Single snapshot, res_2d={}, res_3d={}, n_walkers={}:'.format(RES_2D, RES_3D, N_WALKERS))
print('  forward model      : {:.4f} s'.format(T_FORWARD))
print('  serial, per step   : {:.4f} s'.format(t_per_step_serial))
print('  pooled, per step   : {:.4f} s  ({} processes)'.format(t_per_step_pooled, n_processes))
print()
print('{:>10} {:>16} {:>16}'.format('n_steps', 'serial', 'pooled'))
print('-' * 44)

for n_steps in [1000, 5000, 10000, 20000, 50000, 100000]:
    print('{:>10} {:>16} {:>16}'.format(
        n_steps,
        format_duration(n_steps * t_per_step_serial),
        format_duration(n_steps * t_per_step_pooled)))

print()
print('Caveats, all of which push the true cost UP:')
print('  - walkers were started at the truth, so no burn-in is included;')
print('  - the v_eq-inc and lat_centre-ff degeneracies (header, points 2 and 3) inflate')
print('    the autocorrelation time, so a degenerate posterior needs MORE steps;')
print('  - with only {} data points the posterior is broad, which also lengthens the'
      .format(1 + N_PROFILE_BINS))
print('    chain needed to map its tails.')

In [ ]:
# =============================================================================
# Autocorrelation time: what a production run would actually need
# =============================================================================
# emcee raises if the chain is too short to trust the estimate. We ask for the
# estimate WITHOUT tol so it returns a number anyway, and report it as
# unconverged rather than pretending it is reliable.

tau = sampler.get_autocorr_time(tol=0)

print('{:<18} {:>12} {:>18}'.format('parameter', 'tau [steps]', 'chain/tau'))
print('-' * 50)
for label, t in zip(LABELS, tau):
    print('{:<18} {:>12.1f} {:>18.1f}'.format(label, t, N_STEPS_TIMING / t))

converged = np.all(N_STEPS_TIMING / tau > 50)
print()
print('A chain is normally considered usable at chain/tau > 50: {}'
      .format('MET' if converged else 'NOT met, so tau below is biased low'))
print()

n_steps_needed = int(50 * np.nanmax(tau))
print('Extrapolated production requirement, 50 x max(tau) = {:,} steps'.format(n_steps_needed))
print('  serial : {}'.format(format_duration(n_steps_needed * t_per_step_serial)))
print('  pooled : {}'.format(format_duration(n_steps_needed * t_per_step_pooled)))
print()
print('This is the most useful number here. Re-run with N_STEPS_TIMING raised until the')
print('chain/tau column clears 50, then trust it.')

## Did it recover the input?

A `N_STEPS_TIMING`-step chain: a smoke test of the inversion, not a converged posterior.
Look for whether the truth sits inside the cloud and whether the expected degeneracies show
up as elongated contours.

In [ ]:
# =============================================================================
# Posterior: corner plot and recovery check
# =============================================================================

n_burn = N_STEPS_TIMING // 2
flat_chain = sampler.get_chain(discard=n_burn, flat=True)

print('samples after discarding {} steps of burn-in: {:,}'.format(n_burn, len(flat_chain)))
print()
print('{:<18} {:>12} {:>12} {:>12} {:>10}'.format(
    'parameter', 'truth', 'median', 'std', 'bias/std'))
print('-' * 68)

for i, label in enumerate(LABELS):
    median = np.median(flat_chain[:, i])
    std = np.std(flat_chain[:, i])
    bias = (median - TRUTH[i]) / std if std > 0 else np.nan
    print('{:<18} {:>12.4g} {:>12.4g} {:>12.4g} {:>10.2f}'.format(
        label, TRUTH[i], median, std, bias))

print()
print('If the chain has not converged, a large |bias/std| is not by itself evidence of a')
print('modelling error. Treat it as a flag to re-check only if it persists in a long run.')

fig = corner.corner(flat_chain, labels=LABELS, truths=TRUTH,
                    show_titles=True, title_fmt='.3g')
fig.suptitle('single snapshot, {}-step chain'.format(N_STEPS_TIMING), y=1.02)
plt.show()

In [ ]:
# =============================================================================
# v_eq * sin(inc): the combination the line profile actually constrains
# =============================================================================
# Header point 2. If v_eq and inc are individually poorly constrained but their
# product is tight, that is the degeneracy showing itself, not a sampler failure.

v_sini_chain = flat_chain[:, 1] * np.sin(np.radians(flat_chain[:, 0]))
v_sini_truth = TRUE_V_EQ * np.sin(np.radians(TRUE_INC))

print('{:<22} {:>12} {:>12} {:>14}'.format('quantity', 'truth', 'median', 'frac. scatter'))
print('-' * 64)
print('{:<22} {:>12.1f} {:>12.1f} {:>14.4f}'.format(
    'v_eq [m/s]', TRUE_V_EQ, np.median(flat_chain[:, 1]),
    np.std(flat_chain[:, 1]) / np.median(flat_chain[:, 1])))
print('{:<22} {:>12.1f} {:>12.1f} {:>14.4f}'.format(
    'v_eq sin(inc) [m/s]', v_sini_truth, np.median(v_sini_chain),
    np.std(v_sini_chain) / np.median(v_sini_chain)))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(flat_chain[:, 0], flat_chain[:, 1], s=2, alpha=0.15)
inc_grid = np.linspace(*PRIOR_INC_DEG, 400)[1:]
ax.plot(inc_grid, v_sini_truth / np.sin(np.radians(inc_grid)), 'r-', lw=1.5,
        label='constant v_eq sin(inc)')
ax.plot(TRUE_INC, TRUE_V_EQ, 'r*', ms=16)
ax.set_xlim(flat_chain[:, 0].min() - 2, flat_chain[:, 0].max() + 2)
ax.set_ylim(flat_chain[:, 1].min() - 200, flat_chain[:, 1].max() + 200)
ax.set_xlabel('inc [deg]')
ax.set_ylabel('v_eq [m/s]')
ax.legend()
ax.set_title('does the posterior follow the v sin i ridge?')
fig.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Data vs best fit, to confirm the inversion reproduces the snapshot
# =============================================================================

best_theta = np.median(flat_chain, axis=0)
best_flux, best_profile = forward_model(best_theta)

residual = (PROFILE_DATA - best_profile) / PROFILE_SIGMA

fig, axes = plt.subplots(2, 1, figsize=(6.5, 6), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

axes[0].errorbar(PROFILE_BIN_CENTRES, PROFILE_DATA, yerr=PROFILE_SIGMA,
                 fmt='o', ms=3, lw=1, label='snapshot data')
axes[0].plot(PROFILE_BIN_CENTRES, best_profile, '-', lw=1.5, label='posterior median')
axes[0].set_ylabel('normalised profile')
axes[0].legend()

axes[1].axhline(0.0, color='k', lw=0.8)
axes[1].plot(PROFILE_BIN_CENTRES, residual, 'o-', ms=3, lw=1)
axes[1].set_ylabel('residual / sigma')
axes[1].set_xlabel('Doppler shift from line centre [Angstrom]')

fig.tight_layout()
plt.show()

print('flux excess   data {:.6f}   model {:.6f}   residual {:+.2f} sigma'
      .format(FLUX_DATA, best_flux, (FLUX_DATA - best_flux) / FLUX_SIGMA))
print('profile       reduced chi2 over {} bins: {:.2f}'
      .format(N_PROFILE_BINS, np.sum(residual ** 2) / N_PROFILE_BINS))

In [ ]:
# =============================================================================
# Optional: save the chain
# =============================================================================
# Writes to a NEW file and refuses to overwrite an existing one, per the project
# rule on not clobbering outputs.

SAVE_CHAIN = False    # set True to write

if SAVE_CHAIN:
    out_dir = './outputs/mcmc_timing'
    os.makedirs(out_dir, exist_ok=True)
    out_path = '{}/snapshot_chain_seed{}_{}steps.npz'.format(
        out_dir, RANDOM_SEED, N_STEPS_TIMING)

    if os.path.exists(out_path):
        raise FileExistsError('{} already exists; change RANDOM_SEED or move the old file'
                              .format(out_path))

    np.savez(out_path,
             chain=sampler.get_chain(),
             log_prob=sampler.get_log_prob(),
             truth=TRUTH,
             labels=np.array(LABELS),
             seed=RANDOM_SEED,
             res_2d=RES_2D, res_3d=RES_3D,
             hd_ld_file=HD_LD_FILE, fac_ld_file=FAC_LD_FILE,
             line_wavelength_ang=LINE_WAVELENGTH_ANG,
             flux_data=FLUX_DATA, profile_data=PROFILE_DATA,
             t_per_step_serial=t_per_step_serial,
             t_per_step_pooled=t_per_step_pooled)

    print('written:', out_path)
else:
    print('SAVE_CHAIN is False; nothing written')